# MindHaven-CBT: Fine-Tuning a Specialized Mental Health LLM

Welcome! This notebook walks you through fine-tuning a base LLM (e.g., Llama 3.1 8B or Qwen 2.5 7B) using **QLoRA** (Quantized Low-Rank Adaptation). 

### **Instructions for Google Colab**
1. Change your runtime to a GPU: Go to **Runtime** -> **Change runtime type** -> select **T4 GPU** (or A100/L4 if available).
2. Execute the cells sequentially.

### **Step 1: Install Dependencies**
We need Hugging Face libraries (`transformers`, `peft`, `trl`), datasets, and `bitsandbytes` (for 4-bit quantization).

In [ ]:
!pip install -q trl peft transformers bitsandbytes datasets accelerate

### **Step 2: Load and Preprocess the Dataset**
We will load the **CounselChat** dataset (which contains therapist answers to client questions) and format it into Llama 3.1 / Qwen instruction-following format.

In [ ]:
from datasets import load_dataset

# Load Counsel Chat dataset
dataset = load_dataset("nbertagnolli/counsel-chat", split="train")

# Format function to map columns to chat instruction structure
def format_prompts(batch):
    texts = []
    for q, a in zip(batch["questionText"], batch["answerText"]):
        text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nYou are MindHaven-CBT, an empathetic AI therapist trained in Cognitive Behavioral Therapy.<|eot_id|><|start_header_id|>user<|end_header_id|>\n{q}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n{a}<|eot_id|>"
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)
print("Sample preprocessed text:")
print(dataset[0]["text"])

### **Step 3: Load Model & Tokenizer in 4-bit Quantization**
To fit a large model into Colab's free T4 GPU (16GB VRAM), we load the base model in 4-bit precision. 

*Note: We configure the base model and computation to use `bfloat16` to natively support Llama 3.1 / Qwen 2.5 and bypass PyTorch's float16 gradient scaling errors.*

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"  # Un-gated model (change to Llama 3.1 if you accepted terms)

# Quantization configuration using bfloat16 for computation
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # Use bfloat16
    bnb_4bit_use_double_quant=True
)

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Load Quantized Base Model in BFloat16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,         # Force base weights to bfloat16
    device_map="auto"
)

### **Step 4: Configure LoRA Adapter**
We specify the modules we want to adapt (`q_proj` and `v_proj` inside attention layers).

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

### **Step 5: Setup Trainer and Start Fine-Tuning**
We use **`bf16=True`** (and `fp16=False`) to run training in BFloat16 precision, completely bypassing gradient scaling steps and preventing the `NotImplementedError`.

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "./mindhaven-cbt-lora"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100,
    fp16=False,                         # Disable float16
    bf16=True,                          # Enable bfloat16 mixed precision
    optim="paged_adamw_8bit",
    save_strategy="steps",
    save_steps=50,
    report_to="none",
    dataset_text_field="text",
    max_length=512
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_args,
)

print("Training starting now...")
trainer.train()

# Save adapters
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Adapters saved successfully!")

### **Step 6: Clear Memory & Merge Adapters**
To merge the lightweight adapters with the base model, we must first clear our GPU's VRAM so we don't hit Out-Of-Memory (OOM) crashes, and then reload the base model in high-precision (bfloat16).

In [ ]:
import gc
import torch

# 1. Delete previous model and trainer variables
if 'model' in globals():
    del model
if 'trainer' in globals():
    del trainer

# 2. Force garbage collection and empty CUDA cache
gc.collect()
torch.cuda.empty_cache()
print("GPU cache cleared successfully!")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"  # Ensure this matches your base MODEL_NAME
OUTPUT_DIR = "./mindhaven-cbt-lora"
MERGED_DIR = "./mindhaven-cbt-merged"

print("Reloading base model in bfloat16...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,         # Reload base model in bfloat16
    device_map="auto"
)

print("Merging adapters with base model...")
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
merged_model = model.merge_and_unload()

print(f"Saving final merged model to {MERGED_DIR}...")
merged_model.save_pretrained(MERGED_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.save_pretrained(MERGED_DIR)
print("All done! Model merged and saved successfully.")

### **Step 7 (Optional): Push Merged Model to Hugging Face**
You can upload your final merged model directly to Hugging Face Hub.

In [ ]:
from huggingface_hub import notebook_login

# Login to your Hugging Face account
notebook_login()

In [ ]:
# Push model to hub (replace with your username)
# merged_model.push_to_hub("your-username/mindhaven-cbt-qwen")
# tokenizer.push_to_hub("your-username/mindhaven-cbt-qwen")